### Merger les 2 inputs 
 - Joint les deux fichiers (AlltimePR et WO_all_time) avec WO = WO Ref.
 - Ajoute une colonne NRC? : "NRC" si la colonne Task commence par "NRC", sinon "Normal"
 - Sauvegarde le résultat dans merged_all_PR_WO.xlsx
 - Affiche un résumé

In [1]:
##########################################################################
# Merge les 2 fichiers WO_All_Time.xlsx et AlltimePR_until_260526.xlsx
# utilisant la clé 'WO Ref.' = 'WO'
# Ajout colonne "NRC?" = "NRC" si Task commence par "NRC", sinon "Normal"
#
# Inputs : input_data/WO_All_Time.xlsx
#          input_data/AlltimePR_until_260526.xlsx
# Output : merged_all_PR_WO.xlsx
##########################################################################

import pandas as pd

file1 = "input_data/WO_All_Time.xlsx"
file2 = "input_data/AlltimePR_until_260526.xlsx"
merge_file = "merged_all_PR_WO.xlsx"

df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)
merged = df1.merge(df2, left_on="WO Ref.", right_on="WO", how="left", suffixes=("", "_file2"))

# Ajout nouvelle colonne "NRC?" (NRC ou Normal) 
merged["NRC?"] = merged["Task"].astype(str).str.startswith("NRC").map({True: "NRC", False: "Normal"})

# Final info
merged.to_excel(merge_file, index=False)
print(f"Fichier créé: {merge_file}")
print(f"Colonnes: {len(df1.columns)} → {len(merged.columns)}")
print(f"Lignes: {len(df1)} → {len(merged)}")

print(f"Normal Task: {(merged['NRC?'] == 'Normal').sum()} ({(merged['NRC?'] == 'Normal').sum() / len(merged) * 100:.1f}%)")
print(f"NRC Task: {(merged['NRC?'] == 'NRC').sum()} ({(merged['NRC?'] == 'NRC').sum() / len(merged) * 100:.1f}%)")

Fichier créé: merged_all_PR_WO.xlsx
Colonnes: 33 → 55
Lignes: 85343 → 174499
Normal Task: 96970 (55.6%)
NRC Task: 77529 (44.4%)


In [3]:
###############################################################################
# DESCRIPTION
#
# Ce script analyse un historique de Work Orders (WO) afin d'estimer la
# probabilité de NRC et les Part Numbers (PN) associés pour une liste de
# tâches à planifier.
#
# Entrées :
#   - merged_all_PR_WO.xlsx : historique des WO, MR References, Tasks, PN, Qty
#     et NRC.
#   - input_data/Task_to_plan.xlsx : liste des tâches à analyser (TASK_REF).
#
# Pour chaque tâche, le script :
#   - Recherche les MR References correspondantes.
#   - Calcule les statistiques NRC (lignes NRC, WO avec NRC, probabilité).
#   - Génère des statistiques par MR Reference.
#   - Identifie les PN utilisés dans les tâches NRC ainsi que leurs quantités
#     min, moyennes et max observées.
#
# Sorties :
#   - Un fichier Excel par tâche dans le dossier output_previsions/.
#   - Chaque fichier contient :
#       * un onglet "Résumé" avec les statistiques et prévisions NRC.
#       * un onglet "Détails" avec le détail MR Reference > WO > Task > PN.
#
###############################################################################

import pandas as pd
from pathlib import Path
from tqdm import tqdm

#### Input infos 
merge_file = "merged_all_PR_WO.xlsx"
task_file = "input_data/Task_to_plan.xlsx"
output_base = Path("output_previsions")      # dossier output

#### Lecture du merged file 
df = pd.read_excel(merge_file)
print(f"{len(df)} lignes chargées\n")

for col in ["MR Reference", "Task", "PN", "WO Ref."]:    # processing colonnes importantes
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df["Qty"] = pd.to_numeric(df["Qty"], errors="coerce").fillna(0)  # colonne Qty remplie de 0 si pas de valeurs numeriques


##### Analyser chaque tache 
def analyse_task(input_task, df, output_dir):
    task_df = df[df["MR Reference"].str.startswith(input_task, na=False)].copy()  # regarde specifique tache "input_task"
    mr_refs = sorted(task_df["MR Reference"].dropna().unique())                   # mr ref unique

    if len(mr_refs) == 0:
        print(f"{input_task}: aucune MR Reference trouvée!")
        return

    # Calculs globaux
    total_lignes   = len(task_df)
    nb_wos_uniques = task_df["WO Ref."].nunique()
    nb_lignes_nrc  = int((task_df["NRC?"] == "NRC").sum())
    wos_avec_nrc   = int(task_df.groupby("WO Ref.")["NRC?"].apply(lambda x: (x == "NRC").any()).sum())
    proba_globale  = round(nb_lignes_nrc / total_lignes * 100, 1) if total_lignes > 0 else 0

    # Stats par MR Reference
    mr_stats = (
        task_df.groupby("MR Reference").agg(
            nb_wos=("WO Ref.", "nunique"),                         #nb WO ref unique
            nb_lignes_nrc=("NRC?", lambda x: (x == "NRC").sum())   #nb lignes NRC
        )
        .reset_index()
    )

    # Détails MR Reference > WO > Task > PN
    summary_rows = []
    for mr_ref in mr_refs:
        mr_df = task_df[task_df["MR Reference"] == mr_ref]
        for wo, wo_df in mr_df.groupby("WO Ref."):
            for task, task_group in wo_df.groupby("Task"):
                is_nrc = str(task).upper().startswith("NRC")
                task_type = "NRC" if is_nrc else "Normal"
                
                # addition des qty our chaque PN 
                pn_summary = (
                    task_group.groupby("PN")["Qty"]
                    .sum()
                    .reset_index()
                    .rename(columns={"Qty": "Total_Qty"})
                    .sort_values("PN")
                )
                for _, r in pn_summary.iterrows():
                    if str(r["PN"]).lower() in ("nan", ""):
                        continue
                    summary_rows.append({
                        "MR Reference": mr_ref,
                        "WO Ref.": wo,
                        "Task": task,
                        "Task_Type": task_type,
                        "PN": r["PN"],
                        "Total_Qty": r["Total_Qty"]
                    })

    # Creation dataframe output pour tab "Details"
    summary_df = pd.DataFrame(summary_rows)

    # Garde seulement les lignes NRC
    nrc_df = summary_df[summary_df["Task_Type"] == "NRC"].copy() if not summary_df.empty else pd.DataFrame()

    
    #### Construction du résumé
    def row(cat="", met="", qmin="", qavg="", qmax=""):
        return {"Catégorie": cat, "Métrique": met, "Qté min": qmin, "Qté avg": qavg, "Qté max": qmax}

    resume_rows = []

    # PROBABILITÉ NRC
    resume_rows.append(row("PROBABILITÉ NRC"))
    resume_rows.append(row("Task", input_task))
    resume_rows.append(row("Nb MR References", len(mr_refs)))
    resume_rows.append(row("Nb WOs uniques", nb_wos_uniques))
    resume_rows.append(row("Nb lignes total", total_lignes))
    resume_rows.append(row("Nb lignes NRC", nb_lignes_nrc))
    resume_rows.append(row("Nb WOs avec NRC", wos_avec_nrc))
    resume_rows.append(row("Probabilité NRC (%)", proba_globale))
    resume_rows.append(row())
    resume_rows.append(row())

    # PROBA PAR MR REFERENCE
    resume_rows.append(row("PROBA PAR MR REFERENCE"))
    for _, r in mr_stats.iterrows():
        nb_wos = r["nb_wos"]
        nb_nrc = int(r["nb_lignes_nrc"])
        pct = round(nb_nrc / nb_wos * 100, 1) if nb_wos > 0 else 0
        resume_rows.append(row(r["MR Reference"], f"{nb_nrc} lignes NRC/{nb_wos} WOs      ({pct}%)"))
    resume_rows.append(row())
    resume_rows.append(row())

    # INFOS PNs NRC
    resume_rows.append(row("INFOS PNs NRC", "", "Qté min", "Qté avg", "Qté max"))
    if not nrc_df.empty:
        for pn, pn_group in nrc_df.groupby("PN"):        # loop pour chaque PN unique dans NRC tasks
            wos_with_pn = pn_group["WO Ref."].nunique()  # nb de WOs avec ce PN
            freq = round(wos_with_pn / nb_wos_uniques * 100, 1) if nb_wos_uniques > 0 else 0
            qty_per_wo = pn_group.groupby("WO Ref.")["Total_Qty"].sum()
            resume_rows.append(row(
                cat=pn,
                met=f"{wos_with_pn} NRCs/{nb_wos_uniques} WOs      ({freq}%)",
                qmin=qty_per_wo.min(),
                qavg=round(qty_per_wo.mean(), 2),
                qmax=qty_per_wo.max()
            ))

    resume_df = pd.DataFrame(resume_rows)

    # Création 1 xlsx par task
    output_file = output_dir / f"prevision_{input_task}.xlsx"

    # Création 2 tabs (Résumé et Détails)
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        resume_df.to_excel(writer, sheet_name="Résumé", index=False, header=False)
        if not summary_df.empty:
            summary_df.to_excel(writer, sheet_name="Détails", index=False)

    print(f"{output_file.name} ({nb_wos_uniques} WOs, {nb_lignes_nrc} lignes NRC, {proba_globale}%)")


##### Loop sur toutes les taches du fichier "Task_to_plan.xlsx"
sheets = pd.read_excel(task_file, sheet_name=None)     # lecture des tabs de cet input xlsx

for sheet_name, sheet_df in sheets.items():
    if "TASK_REF" not in sheet_df.columns:
        print(f"Onglet '{sheet_name}' : colonne 'TASK_REF' introuvable, ignoré.")
        continuepct

    tasks = sheet_df["TASK_REF"].dropna().unique()      # task unique dans colonne TASK_REF
    print(f"\nOnglet '{sheet_name}' — {len(tasks)} tâche(s)")

    # Création un dossier par tab
    output_dir = output_base / sheet_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Analyse de chaque task
    for input_task in tqdm(tasks, desc=f"  {sheet_name}"):
        analyse_task(str(input_task).strip(), df, output_dir)

print(f"\nTerminé ! Fichiers dans : {output_base}/")

174499 lignes chargées


Onglet '1535' — 194 tâche(s)


  1535:   0%|                                           | 0/194 [00:00<?, ?it/s]

prevision_231200-DVI-10010-1.xlsx (38 WOs, 33 lignes NRC, 56.9%)


  1535:   1%|▎                                  | 2/194 [00:00<00:10, 17.58it/s]

prevision_271100-DVI-10000-2.xlsx (27 WOs, 20 lignes NRC, 47.6%)
prevision_271100-FUT-10010-1.xlsx (16 WOs, 4 lignes NRC, 23.5%)


...

prevision_ZL-641-GVI-10000-1.xlsx (142 WOs, 420 lignes NRC, 87.5%)
prevision_ZL-121-GVI-10000-1.xlsx (23 WOs, 31 lignes NRC, 57.4%)
prevision_122227-LUB-10000-1.xlsx (51 WOs, 1 lignes NRC, 1.9%)
prevision_273600-OPT-10010-1.xlsx (54 WOs, 3 lignes NRC, 5.5%)


  1538:  98%|████████████████████████████████▏| 169/173 [00:19<00:00, 11.00it/s]

prevision_277141-CHK-10000-1.xlsx (44 WOs, 2 lignes NRC, 4.4%)
prevision_323351-CHK-10000-1.xlsx (53 WOs, 8 lignes NRC, 14.0%)
prevision_327000-FUT-10000-1.xlsx (48 WOs, 29 lignes NRC, 41.4%)
prevision_720000-SDI-15000-1.xlsx (11 WOs, 0 lignes NRC, 0.0%)


  1538: 100%|█████████████████████████████████| 173/173 [00:20<00:00,  8.61it/s]

prevision_731263-RAI-10010-1.xlsx (47 WOs, 0 lignes NRC, 0.0%)
prevision_792362-SDI-10000-1.xlsx (5 WOs, 0 lignes NRC, 0.0%)
prevision_ZL-430-GVI-10000-1.xlsx (96 WOs, 167 lignes NRC, 77.3%)

Terminé ! Fichiers dans : output_previsions/


In [4]:
##########################################################################
# Lecture des fichiers prevision_*.xlsx dans chaque input dossier,
# Extraction des PNs avec NRC et leurs stat (nb WOs, nb NRCs, %, qty),
# Crée\ation fichier summary_[dossier].xlsx par dossier.
#
# Input  : dossiers output_previsions/1535 et output_previsions/1538
# Output : summary_1535.xlsx et summary_1538.xlsx dans chaque dossier
##########################################################################


import pandas as pd
from pathlib import Path
import re

#### Noms des 2 inputs
folders = [
    Path("output_previsions/1535"),
    Path("output_previsions/1538"),
]

#### Extraction infos des fichiers "prevision_*.xlsx"
def parse_prevision_file(filepath):
    wb_sheets = pd.read_excel(filepath, sheet_name=None, header=None)
    resume = wb_sheets.get("Résumé")
    
    # Extraire le nom de la tâche
    task_name = ""
    nb_wos = 0
    rows = resume.values.tolist()

    # Parcourt toutes les lignes (info Task et nb WOs uniques)
    for r in rows:
        if str(r[0]).strip() == "Task":
            task_name = str(r[1]).strip()
        if str(r[0]).strip() == "Nb WOs uniques":
            nb_wos = int(r[1]) if pd.notna(r[1]) else 0

    # Trouver la section "INFOS PNs NRC"
    in_pn_section = False
    pn_rows = []

    for r in rows:
        cat = str(r[0]).strip()
        if cat == "INFOS PNs NRC":
            in_pn_section = True       # activation flag PN section
            continue
        if in_pn_section:
            if cat == "" or pd.isna(r[0]):
                continue
            metrique = str(r[1]).strip() if pd.notna(r[1]) else ""
            # Parse pattern "X NRCs/Y WOs (Z%)" dans 2e colonne 
            match = re.match(r"(\d+)\s+NRCs?/(\d+)\s+WOs", metrique)
            if not match:
                continue
            nb_nrc = int(match.group(1))

            # Lire colonnes des quantites (min, avg, max)
            qmin = r[2] if pd.notna(r[2]) else ""
            qavg = r[3] if pd.notna(r[3]) else ""
            qmax = r[4] if pd.notna(r[4]) else ""
            pourcentage = round(nb_nrc / nb_wos * 100, 1) if nb_wos > 0 else 0

            pn_rows.append({
                "Task": task_name,
                "PN": cat,
                "# WOs Réalisés": nb_wos,
                "# Besoin en NRC": nb_nrc,
                "% de besoin": f"{pourcentage}%",
                "Qty Min": qmin,
                "Qty Avg": qavg,
                "Qty Max": qmax,
            })

    return task_name, pn_rows


#####################################
#### Boucle sur les deux dossiers 
#####################################
for folder in folders:
    # Vérification existence des dossiers 
    if not folder.exists():
        print(f"Dossier introuvable: {folder}")
        continue
    input_files = sorted(folder.glob("prevision_*.xlsx"))
    print(f"\n Fichier {folder.name}: {len(input_files)} inputs")
    
    folder_rows = []
    for file in input_files:
        task_name, nrc_rows = parse_prevision_file(file)
        print(f"  {file.name} → {len(nrc_rows)} PNs NRC")
        folder_rows.extend(nrc_rows)

    summary_df = pd.DataFrame(folder_rows)
    output_file = folder / f"summary_{folder.name}.xlsx"

    if summary_df.empty:
        print(f"  Aucun PN NRC trouvé")
    else:
        summary_df.to_excel(output_file, index=False)
        print(f"--> {output_file.name} — {len(summary_df)} lignes")


 Fichier 1535: 194 inputs
  prevision_122111-CLN-10000-1.xlsx → 11 PNs NRC
  prevision_122227-LUB-10010-1.xlsx → 3 PNs NRC
  prevision_122232-LUB-10000-1.xlsx → 2 PNs NRC
  prevision_122232-LUB-10005-1.xlsx → 60 PNs NRC
  prevision_122232-LUB-10010-1.xlsx → 3 PNs NRC
  prevision_122232-LUB-10015-1.xlsx → 22 PNs NRC
  prevision_122232-LUB-10020-1.xlsx → 2 PNs NRC
  prevision_122232-LUB-10030-1.xlsx → 0 PNs NRC
  prevision_122252-LUB-10025-1.xlsx → 1 PNs NRC
  prevision_122252-LUB-10080-1.xlsx → 0 PNs NRC
  prevision_216000-RAI-10000-2.xlsx → 6 PNs NRC
  prevision_231200-DVI-10000-1.xlsx → 10 PNs NRC
  prevision_231200-DVI-10010-1.xlsx → 17 PNs NRC
  prevision_236000-FUT-10000-1.xlsx → 32 PNs NRC
  prevision_236000-FUT-10010-1.xlsx → 23 PNs NRC
  prevision_237100-OPT-10020-1.xlsx → 2 PNs NRC
  prevision_237131-OPT-10010-1.xlsx → 1 PNs NRC
  prevision_251100-DVI-10000-1.xlsx → 62 PNs NRC
  prevision_254500-OPT-10000-1.xlsx → 35 PNs NRC
  prevision_254521-DVI-10000-1.xlsx → 48 PNs NRC
  p